In [1]:
import sys

print(sys.executable)



C:\MCDI501-Framingham\.venv\Scripts\python.exe


# Evaluación Sumativa 2
## Validación, simulación y métodos de remuestreo

**Asignatura:** MCDI501 — Estadística Computacional para la Toma de Decisiones  
**Proyecto:** Análisis estadístico del riesgo de enfermedad coronaria a diez años  
**Dataset:** Framingham Heart Study  
**Docente:** Jean Paul Maidana González  
**Fase:** 3 — Desarrollo del proyecto

### Objetivo general

Evaluar la estabilidad, incertidumbre y robustez de los resultados
obtenidos en la Sumativa 1 mediante bootstrap no paramétrico,
pruebas de permutación, simulación Monte Carlo y análisis de sensibilidad.

### Vinculación con la Sumativa 1

Se reutilizarán los parámetros estimados, intervalos de confianza,
pruebas de hipótesis y correlaciones previamente calculados
sobre el conjunto de datos Framingham.

## 1. Configuración del entorno de trabajo

En esta sección se importan las bibliotecas necesarias para el análisis
estadístico, la manipulación de datos y la generación de gráficos.

Se establece una semilla aleatoria fija para garantizar la
reproducibilidad de los experimentos.

Las simulaciones y procedimientos de remuestreo utilizarán NumPy y
SciPy. Pandas permitirá manipular el dataset, mientras que Matplotlib
y Seaborn se utilizarán para visualizar los resultados.

In [5]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Semilla para reproducibilidad
SEMILLA = 42

# Generador pseudoaleatorio
rng = np.random.default_rng(SEMILLA)

# Configuración visual de los gráficos
sns.set_theme(style="whitegrid", context="notebook")

print("Bibliotecas cargadas correctamente.")
print("Semilla configurada:", SEMILLA)

Bibliotecas cargadas correctamente.
Semilla configurada: 42


## 2. Localización del proyecto y del dataset

Para mantener la reproducibilidad del análisis, se utilizarán rutas
relativas a la carpeta raíz del repositorio.

El programa buscará automáticamente el archivo `framingham.csv`
en la carpeta `fase-1/datos/raw/`.

También se definirán las rutas donde se almacenarán las figuras y
los resultados de la Sumativa 2.

Los datos originales no serán modificados.

In [6]:
def encontrar_raiz(inicio=None):
    inicio = Path(inicio or Path.cwd()).resolve()

    for candidata in (inicio, *inicio.parents):
        archivo = (
            candidata / "fase-1" / "datos"
            / "raw" / "framingham.csv"
        )

        if archivo.exists():
            return candidata

    raise FileNotFoundError(
        "No se encontró el archivo framingham.csv."
    )


RAIZ = encontrar_raiz()

RUTA_DATOS = (
    RAIZ / "fase-1" / "datos" / "raw" / "framingham.csv"
)

DIR_S2 = RAIZ / "fase-3" / "sumativa-2"
DIR_FIGURAS = DIR_S2 / "figuras"
DIR_RESULTADOS = DIR_S2 / "resultados"

print("Raíz del proyecto:", RAIZ)
print("Ruta del dataset:", RUTA_DATOS)

Raíz del proyecto: C:\MCDI501-Framingham
Ruta del dataset: C:\MCDI501-Framingham\fase-1\datos\raw\framingham.csv


## 3. Carga y verificación inicial del dataset

Se carga el conjunto de datos Framingham desde su ubicación original.

Posteriormente, se crea una copia de trabajo para preservar los datos
originales y se verifica que las dimensiones coincidan con las
documentadas en la Sumativa 1.

El conjunto original contiene 4.238 observaciones y 16 variables.

Esta validación permite comprobar que ambos trabajos utilizan la
misma fuente de información.

In [7]:
# Cargar el dataset original
df = pd.read_csv(RUTA_DATOS)

# Crear una copia para el análisis
df_analisis = df.copy()

# Verificar dimensiones
print("Número de filas:", df_analisis.shape[0])
print("Número de columnas:", df_analisis.shape[1])

# Mostrar las primeras observaciones
display(df_analisis.head())

Número de filas: 4238
Número de columnas: 16


,male,age,education,currentSmoker,cigsPerDay,BPMeds,prevalentStroke,prevalentHyp,diabetes,totChol,sysBP,diaBP,BMI,heartRate,glucose,TenYearCHD
0,1,39,4.0,0,0.0,0.0,0,0,0,195.0,106.0,70.0,26.97,80.0,77.0,0
1,0,46,2.0,0,0.0,0.0,0,0,0,250.0,121.0,81.0,28.73,95.0,76.0,0
2,1,48,1.0,1,20.0,0.0,0,0,0,245.0,127.5,80.0,25.34,75.0,70.0,0
3,0,61,3.0,1,30.0,0.0,0,1,0,225.0,150.0,95.0,28.58,65.0,103.0,1
4,0,46,3.0,1,23.0,0.0,0,0,0,285.0,130.0,84.0,23.10,85.0,85.0,0
